# Inner loop (idea A): a loop over GPT-2's middle layers inside each latent step

Loops layers 4–7 inside each of SIM-CoT GPT-2's six latent steps, adding no cache entries. Plan and diagrams:
`docs/brainstorm/INNER_LOOP_QUICK_REFERENCE.md`.

- **Stage 0** (`STAGE = 0`, evaluation only, about 1–2 hours on a T4):
  - The identity check: the loop with zero new parameters must reproduce the checkpoint on 500/500 questions.
  - Training-free probes: a damped block re-application and a whole-step rerun.
  - SIM-CoT's step-decoder reliability, and the wrong answers split into readout, arithmetic and plan errors.
  - Timing for 100 updates of `ft6` and `loop`.
- **Stage 1** (`STAGE = 1`, the four-arm pilot, under an hour on a G4):
  - Arms `ft6`, `ft12`, `loop` and `loop_k1` on the same 98,304 problems in the same order.
  - Each is evaluated on the 500 validation questions; the loop is swept over k = 1–8.
  - The report cell applies the preregistered go/no-go.
- **Stage 1b** (`STAGE = "1b"`, four arms at batch 128, about 40 minutes on a G4). Stage 1 stopped, and its `ft6` scored
  16 below the main line's answer-only `ft6` (199 vs 215). This stage asks two questions:
  - Is the null the loop's or the recipe's? `ft6_step` is Stage 1's recipe, and `ft6_ce` and `loop_ce` train on the
    answer loss alone.
  - Does idea F's KK-LSE bias help at 6 steps, where there are no copies to cancel? `kk_ce` is `ft6_ce` plus the bias,
    and it is also scored with the bias switched off (`nokk`).

Each finished arm and evaluation goes to Drive, so **rerunning after a disconnect skips finished work**.

**Before running:** Runtime → Change runtime type → **GPU** (a T4 is fine for Stage 0; a G4 or A100 for Stage 1 at
batch 128, or set `BATCH = 32` on a T4), then
Runtime → **Run all**. Keep the tab open until the last cell downloads the results.

In [ ]:
# A run is identified by these settings; rerunning with the same values resumes it.
STAGE = 0                 # 0: checks and probes (evaluation only); 1: the four-arm pilot; "1b": recipe and KK-LSE
ARMS = ["ft6", "ft12", "loop", "loop_k1"]
RECORDS = 98_304          # Stage 1: one pass over these problems per arm (768 updates at batch 128)
BATCH = 128               # learning rates scale by sqrt(BATCH / 32); 32 is the first Stage 1 run, and fits a T4
EVAL_LIMIT = None         # e.g. 50 for a quick look; None scores all 500 validation questions
RUN_TAG = ""              # e.g. "-seed2" with a different SEED for a second seed
SEED = 7
REPO_REF = "main"         # the branch or commit holding scripts/inner_loop

In [ ]:
from pathlib import Path
import json, os, runpy, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
RUN_NAME = f"inner-loop-stage{STAGE}-n{RECORDS}-b{BATCH}-s{SEED}{RUN_TAG}"
BACKUP_DIR = DRIVE / "inner-loop" / RUN_NAME
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

PINNED = BACKUP_DIR / "commit.txt"  # a started run keeps the code it started with
ref = PINNED.read_text().strip() if PINNED.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)],
                   check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert Path("scripts/inner_loop/train.py").exists(), f"{ref} has no scripts/inner_loop; push it first"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
PINNED.write_text(COMMIT)
print("Code:", COMMIT, f"(from {ref})")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = "/content/hf-cache"  # local disk: the school Drive is nearly full
os.environ["TOKENIZERS_PARALLELISM"] = "false"
WORK = Path("/content/inner-loop")  # fast local disk; finished work is copied to BACKUP_DIR
WORK.mkdir(exist_ok=True)
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=BACKUP_DIR / "logs")

run("scripts/setup_data.py", "--eval-only")  # evaluation sets and the pinned GSM8K-Aug arrays, train included
CHECKPOINT = DRIVE / "checkpoints/simcot-gpt2"
run("scripts/latent_gate/fetch_simcot.py", "--output", CHECKPOINT)
EVALS = BACKUP_DIR / "evals"
limit = [] if EVAL_LIMIT is None else ["--limit", EVAL_LIMIT]

def evaluate(name, model_dir, arms, decoder=True):
    target = EVALS / name / "gsm8k_valid.jsonl"
    if target.exists() and EVAL_LIMIT is None:
        print(f"{name}: evaluation already on Drive")
        return
    local = WORK / "evals" / name / "gsm8k_valid.jsonl"
    run("scripts/inner_loop/eval.py", *([] if model_dir is None else ["--checkpoint", model_dir]),
        "--dataset", "gsm8k_valid", "--output", local, "--device", "cuda", "--arms", *arms,
        *(["--decoder"] if decoder else []), *limit)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local, target)

def train(name, records, folder, arm=None, extra=()):
    saved = BACKUP_DIR / folder / name
    if (saved / "model/pytorch_model.bin").exists():
        print(f"{name}: already on Drive")
        return saved
    out = WORK / folder / name
    run("scripts/inner_loop/train.py", "--arm", arm or name, "--checkpoint", CHECKPOINT, "--output", out,
        "--records", records, "--batch", BATCH, "--seed", SEED, "--device", "cuda", "--bf16", *extra)
    shutil.copytree(out / "model", saved / "model", dirs_exist_ok=True)
    for name in ("args.json", "metrics.jsonl"):
        shutil.copy2(out / name, saved / name)
    return saved

In [ ]:
if STAGE == 0:
    # SIM-CoT: identity (loopK with zero parameters), training-free probes, a depth reference, the step decoder.
    evaluate("base-simcot", CHECKPOINT, ["plain", "loop1", "loop2", "loop4", "damp2x0.25", "damp2x0.5", "damp3x0.25",
                                         "rerun2", "rerun3", "plain@12"])
    # CODI: the same identity and probes, without SIM-CoT's decoder.
    evaluate("base-codi", None, ["plain", "loop1", "loop4", "damp2x0.5", "rerun2"], decoder=False)
    # Timing: 100 updates of ft6 and of the loop (k from 1 to 4), to price Stage 1.
    for arm in ("ft6", "loop"):
        train(arm, 100 * BATCH, "timing")
        lines = (BACKUP_DIR / "timing" / arm / "metrics.jsonl").read_text().splitlines()
        print(arm, "100 updates in", json.loads(lines[-1])["seconds"], "s")

In [ ]:
if STAGE == 1:
    SWEEP = {"ft6": ["plain"], "ft12": ["plain", "plain@12"], "loop_k1": ["plain", "loop1"],
             "loop": ["plain"] + [f"loop{k}" for k in range(1, 9)]}
    for arm in ARMS:
        saved = train(arm, RECORDS, "arms")
        evaluate(arm, saved / "model", SWEEP[arm])

In [ ]:
if STAGE == "1b":
    evaluate("base-simcot", CHECKPOINT, ["plain"])  # the reference, in this run's folder
    # name: (train.py arm, extra flags, evaluation arms)
    STAGE_1B = {"ft6_step": ("ft6", [], ["plain"]),
                "ft6_ce": ("ft6", ["--step-weight", 0], ["plain"]),
                "loop_ce": ("loop", ["--step-weight", 0], ["plain"] + [f"loop{k}" for k in range(1, 9)]),
                "kk_ce": ("ft6", ["--step-weight", 0, "--kk"], ["plain", "nokk"])}
    for name, (arm, extra, sweep) in STAGE_1B.items():
        saved = train(name, RECORDS, "arms", arm=arm, extra=extra)
        evaluate(name, saved / "model", sweep)

In [ ]:
run("scripts/inner_loop/report.py", EVALS, "--training", BACKUP_DIR / "arms")

In [ ]:
# The evaluations, training logs and arguments, for the repo's results folder.
from google.colab import files

bundle = Path(f"/content/{RUN_NAME}")
shutil.rmtree(bundle, ignore_errors=True)
if EVALS.exists():
    shutil.copytree(EVALS, bundle / "evals")
for folder in ("arms", "timing"):
    for source in sorted((BACKUP_DIR / folder).glob("*/*.json*")) if (BACKUP_DIR / folder).exists() else []:
        (bundle / folder / source.parent.name).mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, bundle / folder / source.parent.name / source.name)
(bundle / "commit.txt").write_text(COMMIT)
files.download(shutil.make_archive(str(bundle), "zip", bundle))
print(f"Downloading {bundle}.zip. If it does not start, get it from the Files panel on the left.")

**Reading the result**
- **Stage 0.**
  - Each `identity` line must say PASS. If one says FAIL, the loop code is wrong; stop and report it.
  - The `damp` and `rerun` rows are training-free probes, expected to hold about the plain score, not beat it.
  - The step-decoder table and the error split show what SIM-CoT's latents hold.
  - The timing lines price Stage 1.
- **Stage 1.** The verdict line applies the preregistered rule at k = 3:
  - **GO** needs the loop at least 10/500 above the better of `ft6`@6 and `ft12`@12, k=3 at least 5 above k=1 in the
    same model, and the gate off zero.
  - **AMBIGUOUS** (+5 to +9) means one more seed.
  - Anything else is **STOP**.
- **Stage 1b.** The last lines compare the arms at the same thresholds:
  - **Recipe:** `ft6_ce` against `ft6_step`, reported with no verdict.
  - **Loop:** `loop_ce` at k = 3 against `ft6_ce`, with k = 3 against k = 1 as the second check.
  - **KK-LSE:** `kk_ce` against `ft6_ce`, with the bias on against off (`nokk`) as the second check.

**Sharing:** the last cell downloads a zip of the evaluations and training logs. Point Claude at it.